# TOPIC COVERED

In [1]:
from pyspark.sql import SparkSession
spark = SparkSession\
                    .builder\
                    .master("spark://spark-master:7077")\
                    .appName("Day_9_AQE")\
                    .config("spark.sql.warehouse.dir", "/data/spark-warehouse")\
                    .enableHiveSupport()\
                    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/16 19:38:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [13]:
orders_schema_struct = StructType(
                                    [
                                        StructField("order_id",IntegerType()),
                                        StructField("customer_id",IntegerType()),
                                        StructField("product_id",IntegerType()),
                                        StructField("price",FloatType()),
                                        StructField("order_date",DateType()),
                                        StructField("order_status",StringType()),
                                        StructField("state",StringType()),
                                        StructField("quantity",IntegerType()),
                                    ]
                                )

df_orders = spark.read.csv('/data/orders_1gb.csv',schema=orders_schema_struct,header=True)

df_orders.show(5)

+--------+-----------+----------+------+----------+------------+----------------+--------+
|order_id|customer_id|product_id| price|order_date|order_status|           state|quantity|
+--------+-----------+----------+------+----------+------------+----------------+--------+
|       1|     939829|        85|117.77|2008-03-31|   DELIVERED|Himachal Pradesh|       1|
|       2|     980708|        27|198.37|2015-04-30|     SHIPPED|           Delhi|       2|
|       3|     772335|        22|183.63|2017-01-08|   CANCELLED|       Karnataka|       3|
|       4|     105526|        75|169.02|2007-07-04|   DELIVERED|           Delhi|       2|
|       5|     676233|        29|107.84|2003-03-10|   CANCELLED|         Haryana|       1|
+--------+-----------+----------+------+----------+------------+----------------+--------+
only showing top 5 rows



In [14]:
#product table
product_schema_struct = StructType(
                                    [
                                    StructField('product_id',IntegerType()),
                                    StructField('cost_price',FloatType())
                                    ]
                                )

df_product = spark.read.csv('/data/products_12mb.csv',schema=product_schema_struct,header=True)

df_product.show(5)

+----------+----------+
|product_id|cost_price|
+----------+----------+
|         1|    168.37|
|         2|    183.18|
|         3|    155.47|
|         4|    103.91|
|         5|    103.64|
+----------+----------+
only showing top 5 rows



# AQE

In [5]:
df_joined = df_orders.join(df_product,'product_id','inner')

df_joined.write.format('noop').mode('overwrite').save()

#this will create total 3 stages(order file scan& exchange,product file scan & exchange,final join)

In [7]:
df_product_filter = df_product.filter(col('product_id')<=10)
df_product_filter.show()

df2_join = df_orders.join(df_product_filter,'product_id','inner')
df2_join.write.format('noop').mode('overwrite').save()

#original product file size is 12mb ,so it wont go for broadcast as its set for 10mb.
#but even though the size of df_product_filter is very small(less than),its not go for broadcast join and went for shuffe and exchange
#that means spark checking for broadcast join in the evry begining not in runtime

+----------+----------+
|product_id|cost_price|
+----------+----------+
|         1|    168.37|
|         2|    183.18|
|         3|    155.47|
|         4|    103.91|
|         5|    103.64|
|         6|    189.76|
|         7|    132.41|
|         8|    199.79|
|         9|    110.65|
|        10|    158.69|
+----------+----------+



 # Enable Adaptive

In [16]:
spark.conf.set("spark.sql.adaptive.enabled",True)

In [17]:
df_product_filter = df_product.filter(col('product_id')<=10)
#df_product_filter.show()

df2_join = df_orders.join(df_product_filter,'product_id','inner')
df2_join.write.format('noop').mode('overwrite').save()

In [ ]:
orders 
products -> filter -> 100 (<10mb) broadcast ->

# handling data skewed
# salting

In [18]:
#if a product id has lots of data than other ids and we are going for join, that id will go to a single task to execute which will take much time

#here product_id =1 is skewed
spark.conf.set("spark.sql.adaptive.enabled",False)
df_skewed = df_orders.withColumn("product_id",  when(col("product_id")<=90 , 1 ).otherwise(col("product_id")))
df_skewed.groupBy("product_id").count().show()

+----------+--------+
|product_id|   count|
+----------+--------+
|        91|  204093|
|        93|  204501|
|         1|18367397|
|        94|  203765|
|        96|  204504|
|        92|  203406|
|        97|  204278|
|        95|  203570|
|        98|  204486|
+----------+--------+



In [19]:
df_joined = df_skewed.join(df_product, "product_id" , "inner")
df_joined.write.format("noop").mode("overwrite").save()

In [ ]:
aggregation 
group_by product_id 
18 million -> 1 partion -> p1, 18000000

p1,p1_key1 , 1.8 m
p1,p1_key2 , 1.8 m
p1,p1_key3 , 1.8 m
p1,p1_key4 , 1.8 m

customer 

product_id 18 million records
P1
1.8 million records -> P1 -> custom logic ()
1.8 million records -> P1
1.8 million records -> P1
1.8 million records -> P1

1.8 million records -> P1_key10

Salting

spark 3.0
orders (20 million)
order_id, order_date, product
1 , 2026-01-01 , p1
2 , 2026-01-01 , p1

P1 -> 18 million 60 min -> 6mins
2 million 5 min
p1 -> 

1 big job -> 

file 1-> job1 -> filter -> with column (data)
file 2 -> job2  -> narrow  (data)

shuffle 

In [21]:
spark.conf.set("spark.sql.adaptive.enabled",True)
df_joined = df_skewed.join(df_product, "product_id" , "inner")
df_joined.write.format("noop").mode("overwrite").save()

# Executors

In [ ]:
#thin vs fat

In [22]:
spark.stop()

In [23]:
#passing

In [28]:
#from pyspark.sql import SparkSession
spark = (
SparkSession.builder
.master("spark://spark-master:7077")
.appName("executors")
.config("spark.executor.cores", "1")
.config("spark.executor.memory", "1g")
.config("spark.cores.max", "4")
.getOrCreate()
)

# it means each executor has 1 core,1gb ram
#4core/1= 4 executors

In [27]:
spark.stop()

In [26]:
spark = (
SparkSession.builder
.master("spark://spark-master:7077")
.appName("executors")
.config("spark.executor.cores", "2")
.config("spark.executor.memory", "1g")
.config("spark.cores.max", "4")
.getOrCreate()
)

# it means each executor has 2 core,1gb ram
#4/2= total 2 executors



# Dynamic Resource allocation

In [3]:
from pyspark.sql import SparkSession
spark = (SparkSession.
                builder.
                master("spark://spark-master:7077")
                .config("spark.dynamicAllocation.enabled", "true")
                .config("spark.dynamicAllocation.minExecutors", "1")
                .config("spark.dynamicAllocation.initialExecutors","1")
                .config("spark.dynamicAllocation.maxExecutors", "4")
                .config("spark.dynamicAllocation.schedulerBacklogTimeout","10s")
                .config("spark.dynamicAllocation.sustainedSchedulerBacklogTimeout", "1s")
                .config("spark.executor.cores", "1")
                .config("spark.executor.memory", "1g")
                .appName("dataframe").getOrCreate()
                )

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/20 11:37:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/20 11:37:06 WARN Utils: spark.executor.instances less than spark.dynamicAllocation.minExecutors is invalid, ignoring its setting, please update your configs.


In [21]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [22]:
orders_schema_struct = StructType(
                                    [
                                        StructField("order_id",IntegerType()),
                                        StructField("customer_id",IntegerType()),
                                        StructField("product_id",IntegerType()),
                                        StructField("price",FloatType()),
                                        StructField("order_date",DateType()),
                                        StructField("order_status",StringType()),
                                        StructField("state",StringType()),
                                        StructField("quantity",IntegerType()),
                                    ]
                                )

df_orders = spark.read.csv('/data/orders_1gb.csv',schema=orders_schema_struct,header=True)

df_orders.show(5)

[Stage 0:>                                                          (0 + 1) / 1]

+--------+-----------+----------+------+----------+------------+----------------+--------+
|order_id|customer_id|product_id| price|order_date|order_status|           state|quantity|
+--------+-----------+----------+------+----------+------------+----------------+--------+
|       1|     939829|        85|117.77|2008-03-31|   DELIVERED|Himachal Pradesh|       1|
|       2|     980708|        27|198.37|2015-04-30|     SHIPPED|           Delhi|       2|
|       3|     772335|        22|183.63|2017-01-08|   CANCELLED|       Karnataka|       3|
|       4|     105526|        75|169.02|2007-07-04|   DELIVERED|           Delhi|       2|
|       5|     676233|        29|107.84|2003-03-10|   CANCELLED|         Haryana|       1|
+--------+-----------+----------+------+----------+------------+----------------+--------+
only showing top 5 rows



In [23]:
#product table
product_schema_struct = StructType(
                                    [
                                    StructField('product_id',IntegerType()),
                                    StructField('cost_price',FloatType())
                                    ]
                                )

df_product = spark.read.csv('/data/products_12mb.csv',schema=product_schema_struct,header=True)

df_product.show(5)

+----------+----------+
|product_id|cost_price|
+----------+----------+
|         1|    168.37|
|         2|    183.18|
|         3|    155.47|
|         4|    103.91|
|         5|    103.64|
+----------+----------+
only showing top 5 rows



In [8]:
df_join = df_orders.join(df_product,'product_id','inner')

df_join.write.format('noop').mode('overwrite').save()

26/04/20 12:11:35 WARN HeartbeatReceiver: Removing executor 1 with no recent heartbeats: 702777 ms exceeds timeout 120000 ms
26/04/20 12:11:35 ERROR TaskSchedulerImpl: Lost executor 1 on 172.18.0.6: Executor heartbeat timed out after 702777 ms
26/04/20 12:17:26 WARN HeartbeatReceiver: Removing executor 4 with no recent heartbeats: 237234 ms exceeds timeout 120000 ms
26/04/20 12:17:26 ERROR TaskSchedulerImpl: Lost executor 4 on 172.18.0.4: Executor heartbeat timed out after 237234 ms
26/04/20 12:24:15 WARN HeartbeatReceiver: Removing executor 5 with no recent heartbeats: 298711 ms exceeds timeout 120000 ms
26/04/20 12:24:15 ERROR TaskSchedulerImpl: Lost executor 5 on 172.18.0.4: Executor heartbeat timed out after 298711 ms
26/04/20 12:31:26 ERROR TaskSchedulerImpl: Lost executor 6 on 172.18.0.4: worker lost: Not receiving heartbeat for 60 seconds
26/04/20 13:03:25 ERROR TaskSchedulerImpl: Lost executor 7 on 172.18.0.4: worker lost: Not receiving heartbeat for 60 seconds
26/04/20 13:12:5

In [10]:
df_orders_filtered = df_orders.filter(col("product_id")<=1000)
df_orders_filtered.write.format("noop").mode("overwrite").save()

 # .config("spark.dynamicAllocation.enabled", "true")
 #                .config("spark.dynamicAllocation.minExecutors", "1")
 #                .config("spark.dynamicAllocation.initialExecutors","1")
 #                .config("spark.dynamicAllocation.maxExecutors", "4")
 #                .config("spark.dynamicAllocation.schedulerBacklogTimeout","10s")
 #                .config("spark.dynamicAllocation.sustainedSchedulerBacklogTimeout", "1s")
 #                .config("spark.executor.cores", "1")
 #                .config("spark.executor.memory", "1g")

#it means starts with 1 executors,if needed it will increase and max 4 executors
#in each executors 1 core /task can be performed

In [12]:
spark.stop()

In [18]:
from pyspark.sql import SparkSession
spark = (SparkSession.
                builder.
                master("spark://spark-master:7077")
                .config("spark.dynamicAllocation.enabled", "true")
                .config("spark.dynamicAllocation.minExecutors", "1")
                .config("spark.dynamicAllocation.initialExecutors","2") #initial 2 executors
                .config("spark.dynamicAllocation.maxExecutors", "4")
                .config("spark.dynamicAllocation.schedulerBacklogTimeout","10s")
                .config("spark.dynamicAllocation.sustainedSchedulerBacklogTimeout", "1s")
                .config("spark.executor.cores", "2") #each executors can run 2 tasks/core
                .config("spark.executor.memory", "1g")
                .appName("dataframe").getOrCreate()
                )

#initialize order and product df

df_orders_filtered = df_orders.filter(col("product_id")<=1000)
df_orders_filtered.write.format("noop").mode("overwrite").save()

#here initially 2 executors and each has 2 tasks

# schedulerBacklogTimeout , sustainedSchedulerBacklogTimeout
              

In [9]:
spark.stop()

In [10]:
from pyspark.sql import SparkSession
spark = (SparkSession.
                builder.
                master("spark://spark-master:7077")
                .config("spark.dynamicAllocation.enabled", "true")
                .config("spark.dynamicAllocation.minExecutors", "1")
                .config("spark.dynamicAllocation.initialExecutors","2") #initial 1 executors
                .config("spark.dynamicAllocation.maxExecutors", "4")
                .config("spark.dynamicAllocation.schedulerBacklogTimeout","15s")
                .config("spark.dynamicAllocation.sustainedSchedulerBacklogTimeout", "1s")
                .config("spark.executor.cores", "1") #each executors can run 1 tasks/core
                .config("spark.executor.memory", "1g")
                .appName("dataframe1").getOrCreate()
                )

#initialize order and product df

from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

orders_schema_struct = StructType(
                                    [
                                        StructField("order_id",IntegerType()),
                                        StructField("customer_id",IntegerType()),
                                        StructField("product_id",IntegerType()),
                                        StructField("price",FloatType()),
                                        StructField("order_date",DateType()),
                                        StructField("order_status",StringType()),
                                        StructField("state",StringType()),
                                        StructField("quantity",IntegerType()),
                                    ]
                                )

df_orders = spark.read.csv('/data/orders_1gb.csv',schema=orders_schema_struct,header=True)

df_orders.show(5)

#product table
product_schema_struct = StructType(
                                    [
                                    StructField('product_id',IntegerType()),
                                    StructField('cost_price',FloatType())
                                    ]
                                )

df_product = spark.read.csv('/data/products_12mb.csv',schema=product_schema_struct,header=True)

df_product.show(5)

df_orders_filtered = df_orders.filter(col("product_id")<=1000)
df_orders_filtered.write.format("noop").mode("overwrite").save()

26/04/20 14:37:31 WARN Utils: spark.executor.instances less than spark.dynamicAllocation.minExecutors is invalid, ignoring its setting, please update your configs.
                                                                                

+--------+-----------+----------+------+----------+------------+----------------+--------+
|order_id|customer_id|product_id| price|order_date|order_status|           state|quantity|
+--------+-----------+----------+------+----------+------------+----------------+--------+
|       1|     939829|        85|117.77|2008-03-31|   DELIVERED|Himachal Pradesh|       1|
|       2|     980708|        27|198.37|2015-04-30|     SHIPPED|           Delhi|       2|
|       3|     772335|        22|183.63|2017-01-08|   CANCELLED|       Karnataka|       3|
|       4|     105526|        75|169.02|2007-07-04|   DELIVERED|           Delhi|       2|
|       5|     676233|        29|107.84|2003-03-10|   CANCELLED|         Haryana|       1|
+--------+-----------+----------+------+----------+------------+----------------+--------+
only showing top 5 rows



+----------+----------+
|product_id|cost_price|
+----------+----------+
|         1|    168.37|
|         2|    183.18|
|         3|    155.47|
|         4|    103.91|
|         5|    103.64|
+----------+----------+
only showing top 5 rows



26/04/20 14:53:25 ERROR TaskSchedulerImpl: Lost executor 0 on 172.18.0.6: worker lost: Not receiving heartbeat for 60 seconds
26/04/20 15:02:04 ERROR TaskSchedulerImpl: Lost executor 2 on 172.18.0.5: worker lost: Not receiving heartbeat for 60 seconds


# spark submit

In [20]:
#jobs are not run from notebooke,its run from airflow,cicd,shell scripts or cron
#to see those here ,we can use spark-submit
# job.py running

!spark-submit \
--master spark://spark-master:7077 \
--deploy-mode client \
job.py

26/04/20 15:20:55 INFO SparkContext: Running Spark version 3.5.1
26/04/20 15:20:55 INFO SparkContext: OS info Linux, 6.10.14-linuxkit, aarch64
26/04/20 15:20:55 INFO SparkContext: Java version 17.0.15
26/04/20 15:20:55 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/20 15:20:56 INFO ResourceUtils: ==============================================================
26/04/20 15:20:56 INFO ResourceUtils: No custom resources configured for spark.driver.
26/04/20 15:20:56 INFO ResourceUtils: ==============================================================
26/04/20 15:20:56 INFO SparkContext: Submitted application: Day_9_AQE
26/04/20 15:20:56 INFO ResourceProfile: Default ResourceProfile created, executor resources: Map(memory -> name: memory, amount: 1024, script: , vendor: , offHeap -> name: offHeap, amount: 0, script: , vendor: ), task resources: Map(cpus -> name: cpus, amount: 1.0)
26/04/20 15:20:56 INFO Resource

In [21]:
spark.stop()

In [22]:
#spark session is created inside job.py,so no need here
!spark-submit \
--deploy-mode client \
job.py

26/04/20 15:21:43 INFO SparkContext: Running Spark version 3.5.1
26/04/20 15:21:43 INFO SparkContext: OS info Linux, 6.10.14-linuxkit, aarch64
26/04/20 15:21:43 INFO SparkContext: Java version 17.0.15
26/04/20 15:21:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/20 15:21:43 INFO ResourceUtils: ==============================================================
26/04/20 15:21:43 INFO ResourceUtils: No custom resources configured for spark.driver.
26/04/20 15:21:43 INFO ResourceUtils: ==============================================================
26/04/20 15:21:43 INFO SparkContext: Submitted application: Day_9_AQE
26/04/20 15:21:43 INFO ResourceProfile: Default ResourceProfile created, executor resources: Map(memory -> name: memory, amount: 1024, script: , vendor: , offHeap -> name: offHeap, amount: 0, script: , vendor: ), task resources: Map(cpus -> name: cpus, amount: 1.0)
26/04/20 15:21:43 INFO Resource

In [26]:
#no need to hardcode values
spark-submit \
--executor-memory '4g' \
--executor-cores '2' \
--num-executors '4' \
job.py

SyntaxError: invalid syntax (2304811839.py, line 3)

In [27]:
# if spark-submit has data like executor-memory and all and also inside your job u have given those data,
#then spark will give first preference to spark-submit

# deploy-mode

In [ ]:
# terminal 
docker exec -it spark-master bash
#created job.py by cat >job.py then code and cntrl+d
    
# create job.py in master container 
/opt/spark/bin/spark-submit --master spark://spark-master:7077 --deploy-mode client job.py

# working with JSON files

In [30]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.sql.window import Window

spark = SparkSession\
                    .builder\
                    .master("spark://spark-master:7077")\
                    .appName("Day_9_json")\
                    .config("spark.sql.warehouse.dir", "/data/spark-warehouse")\
                    .enableHiveSupport()\
                    .getOrCreate()



26/04/20 16:12:43 WARN Utils: spark.executor.instances less than spark.dynamicAllocation.minExecutors is invalid, ignoring its setting, please update your configs.


In [31]:
df_json = spark.read.json("/data/json_files/sample.json")

df_json.show()

+-------------+---+-------+
|         city| id|   name|
+-------------+---+-------+
|     New York|  1|  Alice|
|San Francisco|  2|    Bob|
|      Chicago|  3|Charlie|
+-------------+---+-------+



# multiline

In [67]:
df_json = spark.read.option('multiLine',True).json("/data/json_files/multi.json")

df_json.show()
df_json.printSchema()

+--------------------+---+------+
|             details| id|  user|
+--------------------+---+------+
|{28, [reading, hi...|101|  jdoe|
|     {34, [cycling]}|102|asmith|
+--------------------+---+------+

root
 |-- details: struct (nullable = true)
 |    |-- age: long (nullable = true)
 |    |-- hobbies: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |-- id: long (nullable = true)
 |-- user: string (nullable = true)



In [69]:
#step-1 hanlde {}
df1= df_json.select('details.*','*').drop('details')
df1.show()

#step-2 handle []
df2 = df1.withColumn('hobbies_explode',explode(col('hobbies'))).drop(col('hobbies'))
df2.show()

+---+-----------------+---+------+
|age|          hobbies| id|  user|
+---+-----------------+---+------+
| 28|[reading, hiking]|101|  jdoe|
| 34|        [cycling]|102|asmith|
+---+-----------------+---+------+

+---+---+------+---------------+
|age| id|  user|hobbies_explode|
+---+---+------+---------------+
| 28|101|  jdoe|        reading|
| 28|101|  jdoe|         hiking|
| 34|102|asmith|        cycling|
+---+---+------+---------------+



In [70]:
# Read Multiline JSON file

df_multi = spark.read.format("json").option("multiLine", True).load("/data/json_files/order_multiline.json")
df_multi.show()

+--------------------+-----------+--------+--------------------+
|             contact|customer_id|order_id|    order_line_items|
+--------------------+-----------+--------+--------------------+
|[9000010000, 9000...|       C001|    O101|[{102.45, I001, 6...|
+--------------------+-----------+--------+--------------------+



In [45]:
df_multi.printSchema()

root
 |-- contact: array (nullable = true)
 |    |-- element: long (containsNull = true)
 |-- customer_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_line_items: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- amount: double (nullable = true)
 |    |    |-- item_id: string (nullable = true)
 |    |    |-- qty: long (nullable = true)



In [73]:
df_multi.show()

#step-1 handle [] in order_line_items
df1= df_multi.withColumn('order_line_items_explode',explode(col('order_line_items')))\
                .drop(col('order_line_items'))

df1.show()

#handle {}
df2= df1.select('*','order_line_items_explode.*')\
        .drop(col('order_line_items_explode'))

df2.show()

+--------------------+-----------+--------+--------------------+
|             contact|customer_id|order_id|    order_line_items|
+--------------------+-----------+--------+--------------------+
|[9000010000, 9000...|       C001|    O101|[{102.45, I001, 6...|
+--------------------+-----------+--------+--------------------+

+--------------------+-----------+--------+------------------------+
|             contact|customer_id|order_id|order_line_items_explode|
+--------------------+-----------+--------+------------------------+
|[9000010000, 9000...|       C001|    O101|       {102.45, I001, 6}|
|[9000010000, 9000...|       C001|    O101|         {2.01, I003, 2}|
+--------------------+-----------+--------+------------------------+

+--------------------+-----------+--------+------+-------+---+
|             contact|customer_id|order_id|amount|item_id|qty|
+--------------------+-----------+--------+------+-------+---+
|[9000010000, 9000...|       C001|    O101|102.45|   I001|  6|
|[90000

# nested json

In [ ]:
{"order_id":100,"order_date":"2026-02-14","customer":{"cust_id":111,"name":"gyan"},"items":[{"item_name":"laptop","price":95000},{"item_name:"bag","price":3000}]}
{"order_id":200,"order_date":"2026-03-15","customer":{"cust_id":233,"name":"xyz"},"items":[{"item_name":"shirt","price":1999},{"item_name:"pant","price":2599}]}

In [60]:
#step-1 read json
df_nested = spark.read.json('/data/json_files/nested.json')

df_nested.show()

+-----------+--------------------+----------+--------+
|   customer|               items|order_date|order_id|
+-----------+--------------------+----------+--------+
|{111, gyan}|[{laptop, 95000},...|2026-02-14|     100|
| {233, xyz}|[{shirt, 1999}, {...|2026-03-15|     200|
+-----------+--------------------+----------+--------+



In [61]:
df_nested.printSchema()

root
 |-- customer: struct (nullable = true)
 |    |-- cust_id: long (nullable = true)
 |    |-- name: string (nullable = true)
 |-- items: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- item_name: string (nullable = true)
 |    |    |-- price: long (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_id: long (nullable = true)



In [64]:
#step-2 handle dictionaries
df_nested.show()


df = df_nested.select('customer.cust_id','customer.name','items','order_date','order_id')

df.show()

#------------ or------------

df2 = df_nested.select('customer.*','*').drop('customer')
df2.show()

+-----------+--------------------+----------+--------+
|   customer|               items|order_date|order_id|
+-----------+--------------------+----------+--------+
|{111, gyan}|[{laptop, 95000},...|2026-02-14|     100|
| {233, xyz}|[{shirt, 1999}, {...|2026-03-15|     200|
+-----------+--------------------+----------+--------+

+-------+----+--------------------+----------+--------+
|cust_id|name|               items|order_date|order_id|
+-------+----+--------------------+----------+--------+
|    111|gyan|[{laptop, 95000},...|2026-02-14|     100|
|    233| xyz|[{shirt, 1999}, {...|2026-03-15|     200|
+-------+----+--------------------+----------+--------+

+-------+----+--------------------+----------+--------+
|cust_id|name|               items|order_date|order_id|
+-------+----+--------------------+----------+--------+
|    111|gyan|[{laptop, 95000},...|2026-02-14|     100|
|    233| xyz|[{shirt, 1999}, {...|2026-03-15|     200|
+-------+----+--------------------+----------+------

In [65]:
#step-3 handle [] 

df_flattern = df.withColumn('items_explode',explode(col('items')))\
                .drop(col('items'))

df_flattern.show()

+-------+----+----------+--------+---------------+
|cust_id|name|order_date|order_id|  items_explode|
+-------+----+----------+--------+---------------+
|    111|gyan|2026-02-14|     100|{laptop, 95000}|
|    111|gyan|2026-02-14|     100|    {bag, 3000}|
|    233| xyz|2026-03-15|     200|  {shirt, 1999}|
|    233| xyz|2026-03-15|     200|   {pant, 2599}|
+-------+----+----------+--------+---------------+



In [66]:
#step-4 handle {}

df_final = df_flattern.select('*','items_explode.*')\
                        .drop('items_explode')

df_final.show()

+-------+----+----------+--------+---------+-----+
|cust_id|name|order_date|order_id|item_name|price|
+-------+----+----------+--------+---------+-----+
|    111|gyan|2026-02-14|     100|   laptop|95000|
|    111|gyan|2026-02-14|     100|      bag| 3000|
|    233| xyz|2026-03-15|     200|    shirt| 1999|
|    233| xyz|2026-03-15|     200|     pant| 2599|
+-------+----+----------+--------+---------+-----+



# complex multiline

In [79]:
df = spark.read.option('multiLine',True).json('/data/json_files/complex_multiline.json')

df.show()

df.printSchema()

#handle [] in data
df1 = df.withColumn('data_explode',explode(col('data')))\
            .drop(col('data'))
df1.show()



+--------------------+--------+-------+
|                data|messages|success|
+--------------------+--------+-------+
|[{1111, [{1, Exam...|      []|   true|
+--------------------+--------+-------+

root
 |-- data: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- MainId: long (nullable = true)
 |    |    |-- categories: array (nullable = true)
 |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |-- CategoryID: long (nullable = true)
 |    |    |    |    |-- CategoryName: string (nullable = true)
 |    |    |-- firstName: string (nullable = true)
 |    |    |-- lastName: string (nullable = true)
 |-- messages: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- success: boolean (nullable = true)

+--------+-------+--------------------+
|messages|success|        data_explode|
+--------+-------+--------------------+
|      []|   true|{1111, [{1, Examp...|
|      []|   true|{122, [{2, Exampl...|
|  

In [80]:
#handle {} in data_explode

df2 = df1.select('data_explode.*','*').drop(col('data_explode'))
df2.show()



+------+---------------+---------+--------+--------+-------+
|MainId|     categories|firstName|lastName|messages|success|
+------+---------------+---------+--------+--------+-------+
|  1111| [{1, Example}]|    bunny|    dgr8|      []|   true|
|   122|[{2, Example2}]|     prem|   kumar|      []|   true|
|   133|[{3, Example3}]|      sai|     ram|      []|   true|
|   144|[{4, Example4}]|   pramod|   kumar|      []|   true|
+------+---------------+---------+--------+--------+-------+



In [94]:
#handle [] in categories,the handle {}

df3 = df2.withColumn('categories_explode',explode(col('categories')))\
            .drop(col('categories'))\
            .select('categories_explode.*','*')\
            .drop(col('categories_explode'))

df3.show()

+----------+------------+------+---------+--------+--------+-------+
|CategoryID|CategoryName|MainId|firstName|lastName|messages|success|
+----------+------------+------+---------+--------+--------+-------+
|         1|     Example|  1111|    bunny|    dgr8|      []|   true|
|         2|    Example2|   122|     prem|   kumar|      []|   true|
|         3|    Example3|   133|      sai|     ram|      []|   true|
|         4|    Example4|   144|   pramod|   kumar|      []|   true|
+----------+------------+------+---------+--------+--------+-------+



In [122]:
df_final = df3.withColumn('success',\
                          when(col('success')=='true','Pass')\
                          .when(col('success')=='NA','Pass')\
                          .otherwise("fail")
                         )\
            
                       

df_final.show()

df_final.filter(size(col('messages'))==0).show()




+----------+------------+------+---------+--------+--------+-------+
|CategoryID|CategoryName|MainId|firstName|lastName|messages|success|
+----------+------------+------+---------+--------+--------+-------+
|         1|     Example|  1111|    bunny|    dgr8|      []|   Pass|
|         2|    Example2|   122|     prem|   kumar|      []|   Pass|
|         3|    Example3|   133|      sai|     ram|      []|   Pass|
|         4|    Example4|   144|   pramod|   kumar|      []|   Pass|
+----------+------------+------+---------+--------+--------+-------+

+----------+------------+------+---------+--------+--------+-------+
|CategoryID|CategoryName|MainId|firstName|lastName|messages|success|
+----------+------------+------+---------+--------+--------+-------+
|         1|     Example|  1111|    bunny|    dgr8|      []|   Pass|
|         2|    Example2|   122|     prem|   kumar|      []|   Pass|
|         3|    Example3|   133|      sai|     ram|      []|   Pass|
|         4|    Example4|   144| 

In [109]:
x = df_final.select('messages').collect()[0][0]
type(x) ==list

True

In [121]:
#issue in datatype ,message column is array string
df4 = df_final.withColumn('messages',\
                       when(size(col('messages'))==0,'NULL')\
                        .otherwise(col('messages'))
                       )

df4.show() 

AnalysisException: [DATATYPE_MISMATCH.DATA_DIFF_TYPES] Cannot resolve "CASE WHEN (size(messages) = 0) THEN NULL ELSE messages END" due to data type mismatch: Input to `casewhen` should all be the same type, but it's ["STRING", "ARRAY<STRING>"].;
'Project [CategoryID#2407L, CategoryName#2408, MainId#2111L, firstName#2113, lastName#2114, CASE WHEN (size(messages#2064, true) = 0) THEN NULL ELSE messages#2064 END AS messages#2808, success#2735]
+- Project [CategoryID#2407L, CategoryName#2408, MainId#2111L, firstName#2113, lastName#2114, messages#2064, CASE WHEN (success#2065 = cast(true as boolean)) THEN Pass WHEN (success#2065 = cast(NA as boolean)) THEN Pass ELSE fail END AS success#2735]
   +- Project [CategoryID#2407L, CategoryName#2408, MainId#2111L, firstName#2113, lastName#2114, messages#2064, success#2065]
      +- Project [categories_explode#2393.CategoryID AS CategoryID#2407L, categories_explode#2393.CategoryName AS CategoryName#2408, MainId#2111L, firstName#2113, lastName#2114, 

In [133]:
#issue in datatype ,message column is array string
df4 =  df_final.withColumn('messages',col('messages').cast('string'))
                       

df4.show()
df4.printSchema()
df5 = df4.withColumn('messages',\
                                  when(col('messages')=='[]',lit(None))\
                                  .otherwise(col('messages'))
                    )
df5.show()
#----------------------OR--------------------
#dont type cast
df_4 = df_final.withColumn('messages',\
                                  when(size(col('messages'))==0,lit(None))\
                                  .otherwise(col('messages'))
                    )

df_4.show()


+----------+------------+------+---------+--------+--------+-------+
|CategoryID|CategoryName|MainId|firstName|lastName|messages|success|
+----------+------------+------+---------+--------+--------+-------+
|         1|     Example|  1111|    bunny|    dgr8|      []|   Pass|
|         2|    Example2|   122|     prem|   kumar|      []|   Pass|
|         3|    Example3|   133|      sai|     ram|      []|   Pass|
|         4|    Example4|   144|   pramod|   kumar|      []|   Pass|
+----------+------------+------+---------+--------+--------+-------+

root
 |-- CategoryID: long (nullable = true)
 |-- CategoryName: string (nullable = true)
 |-- MainId: long (nullable = true)
 |-- firstName: string (nullable = true)
 |-- lastName: string (nullable = true)
 |-- messages: string (nullable = true)
 |-- success: string (nullable = false)

+----------+------------+------+---------+--------+--------+-------+
|CategoryID|CategoryName|MainId|firstName|lastName|messages|success|
+----------+----------